In [ ]:
from pathlib import Path

from ai_tools.data_extractor.tools import extract_funcs as ef

import re

In [ ]:
save_data = Path('all_data') / 'processed'

RAW ICT283 DATA

In [ ]:
ict_data_pth = Path().cwd() / 'ai_tools' / 'data_extractor' / 'data' / 'raw'

txt_files = ef.search_txt_files(ict_data_pth)

In [ ]:
def preamble_extract(pth):
    
    with open(pth, 'r', encoding='utf-8') as f:
        
        lines = ""
        temp = f.readline()
        
        while not re.search(r"^([Qq].?)$", temp.strip()):
            lines += temp
            temp = f.readline()
            
        lines = re.sub(r"^(-+)$", "", lines,flags=re.MULTILINE)
        lines = lines.strip()
        lines += '\n'
        
    return lines 

In [ ]:
def combine_and_refactor_files(list : list[str], save_pth : str):
    
    to_write = ""
    
    for pth in list:
        
        preamble = re.sub(r'\n+',' ', preamble_extract(pth))
        preamble = preamble if len(preamble) > 0 else "new topic"
        
        to_write += "Context: " + preamble + '\n'
        
        for pairs in ef.extract_qna_pairs([pth]):
                
            refactored_pair = f'Question: {pairs['Q'].strip()} Answer: {pairs['A'].strip()}'
            refactored_pair = re.sub(r'\n+',' ', refactored_pair)
            refactored_pair = refactored_pair.strip() + '\n'
            to_write += refactored_pair
            
    with open(save_pth, 'w') as f:
        
        f.write(to_write)
        
        
    return to_write

combine_and_refactor_files(txt_files, save_data / 'ICT283_cleaned_raws.txt')

STACK OVERFLOW DATA

In [ ]:
stack_oflow_path = Path().cwd() / 'ai_tools' / 'data_preprocessing' / 'processed_data' / 'train_ready.parquet'
print('exists') if stack_oflow_path.exists() and stack_oflow_path.is_file() else print('does not exist')

In [ ]:
import pandas as pd

oflow_df = pd.read_parquet(stack_oflow_path)
oflow_df.head()

In [ ]:
def make_single_line(string_in):
    
    string_out = re.sub(r'\n+', ' ', string_in)
    string_out = string_out.strip()
    
    return string_out

def sample_of_oflow_df(df, n = 100):
    
    new_df = oflow_df.groupby(by=['topic']).sample(n=n).drop(columns=['label'])
    singled_df = new_df.map(make_single_line)
    
    return singled_df

def  create_oflow_txt(df : pd.DataFrame, save_pth : str):
    
    to_write = ""
    for _, row in df.iterrows():
        
        topic : str = row['topic']
        question_title : str = row['question_title']
        question_body : str = row['question_body']
        answer_body : str = row['answer_body']
        
        row_formatted = f'Topic: {topic.lower()} Question: {question_title} {question_body} Answer: {answer_body}'
        row_formatted += '\n'
        
        to_write += row_formatted
        
    with open(str(save_pth), 'w') as f:
        
        f.write(to_write)

new_oflow_df = sample_of_oflow_df(oflow_df)
create_oflow_txt(new_oflow_df, save_data / 'stack_overflow_data.txt')